# 📈 Linear Regression: Simple & Multiple
### From-scratch `Mylr` class 🧑‍💻 + scikit-learn 🧰 + evaluation metrics 📏

---

## 📑 Table of Contents
1. 📘 Introduction — what is Linear Regression?
2. 🧮 The math behind it
3. 🛠️ Setup — imports and helper function
4. 🎓 Part A — Simple Linear Regression (placement data)
5. 🏠 Part B — Multiple Linear Regression (2 features, 3D)
6. ⚖️ Comparison: Simple vs Multiple
7. 📝 Notes on code improvements
8. 🏁 Summary

## 📘 1. Introduction

**Regression** is a supervised-learning task where we predict a **continuous number** (salary, price, temperature, ...) from one or more input features.

**Linear Regression** is the most fundamental regression algorithm. It assumes that the target depends on the inputs through a **straight line** (or a flat plane / hyperplane when there are many inputs), and it finds the line that fits the data best.

There are two flavours covered in this notebook:

| | 🎓 Simple Linear Regression | 🏠 Multiple Linear Regression |
|---|---|---|
| **Inputs** | **1** feature | **2 or more** features |
| **Shape of the fit** | A line 📏 | A plane / hyperplane 🛩️ |
| **Example** | CGPA → Package (LPA) | Area + Rooms → House price |
| **Equation** | `y = m·x + b` | `y = b₀ + b₁x₁ + b₂x₂ + …` |

📌 **Where is it used?** Salary / price prediction, sales forecasting, risk estimation, trend analysis, and as a **baseline model** before trying complex algorithms.

## 🧮 2. The math behind it

### 🎓 Simple Linear Regression
Model:

$$\hat{y} = m\,x + b$$

"Best fit" means the line that **minimizes the sum of squared errors** (Ordinary Least Squares, OLS):

$$\text{Loss} = \sum_{i=1}^{n}\left(y_i-\hat{y}_i\right)^2$$

Setting the derivatives to zero gives a **closed-form solution** (no iterations needed!):

$$\boxed{m=\frac{\sum (x_i-\bar{x})(y_i-\bar{y})}{\sum (x_i-\bar{x})^2}} \qquad\qquad \boxed{b=\bar{y}-m\,\bar{x}}$$

### 🏠 Multiple Linear Regression
Model:

$$\hat{y} = b_0 + b_1x_1 + b_2x_2 + \dots + b_kx_k$$

In matrix form the OLS solution is the **Normal Equation**:

$$\boldsymbol{\beta}=(X^{T}X)^{-1}X^{T}y$$

💡 *Each coefficient `bⱼ` means: "if feature `xⱼ` increases by 1 unit **while all other features are held constant**, the prediction changes by `bⱼ`".*

### ✅ Assumptions of Linear Regression
1. 📏 **Linearity** — the relationship between inputs and target is linear
2. 🔗 **Independence** — observations are independent of each other
3. 📊 **Homoscedasticity** — the spread of errors is roughly constant
4. 🔔 **Normality of residuals** — errors are roughly normally distributed
5. 🚫 **No multicollinearity** (multiple regression) — input features should not be highly correlated with each other

## 🛠️ 3. Setup — imports and helper function

All libraries are imported once. We also fix the random seed so results are **reproducible** 🔁.

### 📏 One helper for all metrics
Both parts of this notebook need the same five evaluation metrics, so we write the function **once** and reuse it. 

| Metric | Meaning | Good value |
|---|---|---|
| **MAE** — Mean Absolute Error | Average size of the error, in the *same unit* as the target. Easy to read, not too sensitive to outliers | Lower ✅ |
| **MSE** — Mean Squared Error | Average of squared errors. Punishes big mistakes heavily; unit is squared | Lower ✅ |
| **RMSE** — Root MSE | √MSE → back in the original unit, but still penalizes big errors | Lower ✅ |
| **R²** — Coefficient of determination | Share of the variance in `y` explained by the model (1 = perfect, 0 = as bad as predicting the mean) | Closer to 1 ✅ |
| **Adjusted R²** | R² corrected for the number of features. It only rises if a new feature really helps | Closer to 1 ✅ |

$$R^2 = 1-\frac{\sum (y-\hat{y})^2}{\sum (y-\bar{y})^2} \qquad\qquad \text{Adj. }R^2 = 1-\frac{(1-R^2)(n-1)}{n-k-1}$$

where `n` = number of samples and `k` = number of features. (In the original notebook `n=40` and `k=1` were typed by hand; here they are calculated automatically.)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)                      # reproducibility
plt.style.use("seaborn-v0_8-whitegrid")


def evaluate(y_true, y_pred, n_features):
    """Return MAE, MSE, RMSE, R2 and Adjusted R2 as a dictionary."""
    n = len(y_true)
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2,
        "Adjusted R2": 1 - ((1 - r2) * (n - 1) / (n - n_features - 1)),
    }

**🔍 Code explanation — `evaluate`**  
It takes the true values, the predictions and the number of features `k`, and returns all five metrics in one dictionary. The last line is the adjusted-R² formula from above, with `n = len(y_true)` taken automatically.

---
# 🎓 Part A — Simple Linear Regression

### 📖 Definition
**Simple Linear Regression** models the relationship between **one input feature (`x`)** and **one target (`y`)** with a straight line `y = m·x + b`.  
- `m` = **slope** → how much `y` changes when `x` increases by 1  
- `b` = **intercept** → the predicted `y` when `x = 0`

### 🕒 When is it used?
- There is **one clear predictor** and the scatter plot looks roughly like a straight line
- You need a quick, interpretable **baseline**
- Examples: CGPA → placement package, experience → salary, study hours → marks

### ✅ Advantages
- 🧠 Extremely simple and easy to explain
- ⚡ Very fast — has a closed-form solution (no training loop)
- 🔍 Fully interpretable (slope and intercept have a clear meaning)
- 📉 Needs very little data and is hard to overfit

### ❌ Disadvantages
- 📏 Only captures **linear** patterns
- 1️⃣ Uses only one feature, so it ignores everything else that influences the target
- 🚨 Sensitive to **outliers**
- ⚠️ Predictions outside the range of the training data (extrapolation) can be unreliable

## 📂 A1. Load the data

Dataset: **`placement.csv`** — CGPA of students (`cgpa`) and the package they got (`package`, in LPA).

📌 The cell below first looks for `placement.csv` (in Colab: upload it to `/content/`). 

⚠️ **If the file is not found**, it creates a similar *synthetic* dataset so that the notebook still runs end to end. A message tells you which one is being used.

In [ ]:
df = None
for path in ["/content/placement.csv", "placement.csv"]:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"✅ Loaded real data from: {path}")
        break

if df is None:
    print("⚠️ placement.csv not found -> using SYNTHETIC data with a similar pattern. "
          "Upload the real file to /content/placement.csv for your own results.")
    rng = np.random.default_rng(42)
    cgpa = np.clip(rng.normal(6.9, 1.0, 200), 4.0, 10.0).round(2)
    package = (0.558 * cgpa - 0.896 + rng.normal(0, 0.31, 200)).round(2)
    df = pd.DataFrame({"cgpa": cgpa, "package": package})

print("Shape:", df.shape)
df.head()

## 👀 A2. Visualize the data

Always plot before modelling! If the points follow a rough straight line, simple linear regression is a sensible choice.

In [ ]:
plt.scatter(df["cgpa"], df["package"], color="teal", alpha=0.7)
plt.xlabel("CGPA")
plt.ylabel("Package (LPA)")
plt.title("CGPA vs Package")
plt.show()

✅ The data is roughly **linear**: higher CGPA tends to give a higher package.

## ✂️ A3. Split into train and test

- `x` must be **2-D** `(n_samples, n_features)` → `df[["cgpa"]]` (double brackets); we use `.values` to get a plain NumPy array
- `y` is the 1-D target
- `test_size=0.2` → 80 % for training, 20 % for testing
- `random_state=2` → the same split every time

In [ ]:
x = df[["cgpa"]].values
y = df["package"].values

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)
print("Train:", x_train.shape, "| Test:", x_test.shape)

## 🧰 A4. Using scikit-learn's `LinearRegression`

`fit` learns the slope and intercept from the training data. After fitting they are stored in `coef_` (slope) and `intercept_`. 

🔑 Because we pass NumPy arrays (not DataFrames) there is no *"X does not have valid feature names"* warning.

In [ ]:
lr_sk = LinearRegression()
lr_sk.fit(x_train, y_train)

print("Slope (m)     :", lr_sk.coef_[0])
print("Intercept (b) :", lr_sk.intercept_)

# predict for a single student (the first test row)
print("\nCGPA =", x_test[0][0], "-> predicted package =", lr_sk.predict(x_test[0].reshape(1, -1))[0])

📌 **How to read it:** on the original placement data the slope was about **0.56** → *every +1 CGPA adds roughly 0.56 LPA to the predicted package*. The intercept (about −0.90) is only the mathematical value at CGPA = 0, which is outside the real data range, so it has no practical meaning on its own.

### 📉 The best-fit line

In [ ]:
plt.scatter(df["cgpa"], df["package"], color="teal", alpha=0.6, label="data")
plt.plot(x_train, lr_sk.predict(x_train), color="red", lw=2, label="best-fit line")
plt.xlabel("CGPA")
plt.ylabel("Package (LPA)")
plt.title("Simple Linear Regression fit")
plt.legend()
plt.show()

## 🧑‍💻 A5. Our own `Mylr` class — explained in depth

Now we build the same model **from scratch** using the closed-form formulas from Section 2.

**`__init__`** — creates the two things the model will learn: slope `m` and intercept `b` (both `None` until `fit` is called).

**`fit`** — the core:

| Line | What it does |
|---|---|
| `x = x_train.ravel()`, `y = y_train.ravel()` | Flattens the arrays to 1-D so that `x[i]` is a single number |
| `x_mean`, `y_mean` | The averages x̄ and ȳ — calculated **once** before the loop |
| `num += (x[i] - x_mean) * (y[i] - y_mean)` | Builds the **numerator** Σ(xᵢ − x̄)(yᵢ − ȳ) one row at a time |
| `den += (x[i] - x_mean) ** 2` | Builds the **denominator** Σ(xᵢ − x̄)² |
| `self.m = num / den` | Slope from the formula |
| `self.b = y_mean - self.m * x_mean` | Intercept from the formula — the line always passes through the point (x̄, ȳ) |

**`predict`** — applies `m·x + b` to every input row.

💡 There is **no learning rate and no epochs** here. Unlike gradient descent, least squares gives the exact answer in a single pass.

In [ ]:
class Mylr:
    def __init__(self):
        self.m = None
        self.b = None

    def fit(self, x_train, y_train):
        x = x_train.ravel()
        y = y_train.ravel()
        x_mean, y_mean = x.mean(), y.mean()      # computed once, not inside the loop

        num = 0
        den = 0
        for i in range(x.shape[0]):
            num += (x[i] - x_mean) * (y[i] - y_mean)
            den += (x[i] - x_mean) ** 2

        self.m = num / den
        self.b = y_mean - self.m * x_mean
        print("Slope (m)     :", self.m)
        print("Intercept (b) :", self.b)

    def predict(self, x_test):
        return self.m * x_test.ravel() + self.b

In [ ]:
my_lr = Mylr()
my_lr.fit(x_train, y_train)

print("\nCGPA =", x_test[0][0], "-> predicted package =", my_lr.predict(x_test[0])[0])

✅ **Check:** the slope, intercept and prediction from `Mylr` are **identical** to scikit-learn's (see A4). Our from-scratch class reproduces the library result exactly.

In [ ]:
print("Same slope?     ", np.isclose(my_lr.m, lr_sk.coef_[0]))
print("Same intercept? ", np.isclose(my_lr.b, lr_sk.intercept_))

## 📏 A6. Evaluate the model

We predict on the **test set** (data the model has never seen) and use the `evaluate` helper. Here `n_features = 1`.

In [ ]:
y_pred_simple = my_lr.predict(x_test)
metrics_simple = evaluate(y_test, y_pred_simple, n_features=1)

pd.DataFrame(metrics_simple, index=["Simple LR (own Mylr)"]).T.round(4)

🔎 **Reading the results** (original run: MAE ≈ 0.29 LPA, RMSE ≈ 0.35 LPA, R² ≈ 0.78, adjusted R² ≈ 0.77):
- On average the prediction is off by about **0.3 LPA** (MAE) → quite good
- **R² ≈ 0.78** → about 78 % of the variation in package is explained by CGPA alone
- Adjusted R² is only slightly lower than R², which is expected with just one feature

---
# 🏠 Part B — Multiple Linear Regression

### 📖 Definition
**Multiple Linear Regression** predicts the target from **two or more input features** by fitting a **hyperplane**:

$$\hat{y}=b_0+b_1x_1+b_2x_2+\dots+b_kx_k$$

### 🕒 When is it used?
- The target is influenced by **several factors at once**
- You want to know the **individual effect of each factor**, with the others held constant
- Examples: house price (area, rooms, age), salary (experience, education, skills), sales (price, advertising, season)

### ✅ Advantages
- 🎯 Usually more accurate than simple regression because it uses more information
- 🔍 Still interpretable — one coefficient per feature
- ⚡ Fast to train; also has a closed-form solution
- 📊 Reveals which features matter most

### ❌ Disadvantages
- 🔗 **Multicollinearity** — highly correlated features make coefficients unstable
- 🧮 Adding useless features can **overfit** (R² never decreases when features are added — this is why **adjusted R²** exists)
- 📏 Still assumes a **linear** relationship
- 🚨 Sensitive to outliers and needs feature scaling if you compare coefficient sizes

## 🧪 B1. Create a 2-feature dataset

`make_regression` generates synthetic data where the target truly depends on 2 features plus random noise (`noise=50`).

`coef=True` additionally returns the **true coefficients** used to generate the data, so later we can check how close our model gets. 🎯 (`random_state` makes the data reproducible.)

In [ ]:
x_m, y_m, true_coef = make_regression(n_samples=100, n_features=2, n_informative=2,
                                      n_targets=1, noise=50, coef=True, random_state=42)

df_m = pd.DataFrame({"Feature1": x_m[:, 0], "Feature2": x_m[:, 1], "target": y_m})
df_m.head()

### 🌐 Visualize in 3D
With two features we need three axes: Feature1, Feature2 and the target. The points should form a tilted, noisy **plane**. (The plot is interactive — rotate it with the mouse! 🖱️)

In [ ]:
fig = px.scatter_3d(df_m, x="Feature1", y="Feature2", z="target", opacity=0.8,
                    title="3D view of the data")
fig.show()

## ✂️ B2. Split and train

Same steps as Part A — the code does not change even though there are now 2 features, because scikit-learn handles any number of columns in `x`.

In [ ]:
x_train_m, x_test_m, y_train_m, y_test_m = train_test_split(x_m, y_m, test_size=0.2, random_state=3)

lr_multi = LinearRegression()
lr_multi.fit(x_train_m, y_train_m)

print("Coefficients (b1, b2):", lr_multi.coef_)
print("Intercept (b0)       :", lr_multi.intercept_)

🔎 **Reading the coefficients:** `b1` tells how much the target changes for +1 in Feature1 **keeping Feature2 fixed**, and `b2` the same for Feature2. The intercept is the prediction when both features are 0.

### 🎯 Did the model recover the truth?
Compare the learned coefficients with the true ones that generated the data.

In [ ]:
pd.DataFrame({"True coefficient": true_coef,
              "Learned coefficient": lr_multi.coef_},
             index=["Feature1", "Feature2"]).round(3)

✅ The learned values are close to the true ones. They are not exactly equal because of the random noise added to the data (and only 80 training rows) — more data or less noise brings them even closer.

## 📏 B3. Predict and evaluate

⚠️ In the original notebook the predictions were saved in the variable `y`, which **overwrote the target** by mistake. Here they are stored in `y_pred_multi`. We evaluate with `n_features = 2`.

In [ ]:
y_pred_multi = lr_multi.predict(x_test_m)
metrics_multi = evaluate(y_test_m, y_pred_multi, n_features=2)

pd.DataFrame(metrics_multi, index=["Multiple LR (sklearn)"]).T.round(4)

### 🛩️ The fitted plane in 3D
The model's prediction surface is a flat plane. The dots are the real data points; the semi-transparent sheet is what the model predicts for every (Feature1, Feature2) pair. 

**🔍 Code explanation:** `np.meshgrid` builds a grid over the range of both features, then we predict the target for every grid point with `predict` and draw it with `go.Surface`.

In [ ]:
g1 = np.linspace(x_m[:, 0].min(), x_m[:, 0].max(), 20)
g2 = np.linspace(x_m[:, 1].min(), x_m[:, 1].max(), 20)
G1, G2 = np.meshgrid(g1, g2)
Z = lr_multi.predict(np.c_[G1.ravel(), G2.ravel()]).reshape(G1.shape)

fig = go.Figure(data=[
    go.Scatter3d(x=df_m["Feature1"], y=df_m["Feature2"], z=df_m["target"],
                 mode="markers", marker=dict(size=4, opacity=0.8), name="data"),
    go.Surface(x=G1, y=G2, z=Z, opacity=0.5, showscale=False, name="fitted plane"),
])
fig.update_layout(title="Multiple Linear Regression: fitted plane",
                  scene=dict(xaxis_title="Feature1", yaxis_title="Feature2", zaxis_title="target"))
fig.show()

### 🔄 Actual vs Predicted
A quick sanity check that works for **any** number of features: if the model is good, the points lie close to the red diagonal (prediction = actual).

In [ ]:
plt.scatter(y_test_m, y_pred_multi, color="purple", alpha=0.7)
lims = [min(y_test_m.min(), y_pred_multi.min()), max(y_test_m.max(), y_pred_multi.max())]
plt.plot(lims, lims, color="red", lw=2)
plt.xlabel("Actual target")
plt.ylabel("Predicted target")
plt.title("Actual vs Predicted (test set)")
plt.show()

---
## ⚖️ 6. Comparison: Simple vs Multiple Linear Regression

### 📊 Conceptual comparison

| Feature 🔍 | 🎓 Simple Linear Regression | 🏠 Multiple Linear Regression |
|---|---|---|
| **Number of inputs** | 1 | 2 or more |
| **Equation** | `y = m·x + b` | `y = b₀ + b₁x₁ + … + bₖxₖ` |
| **Fitted shape** | Straight line 📏 | Plane / hyperplane 🛩️ |
| **Parameters learned** | 2 (`m`, `b`) | k + 1 (one per feature + intercept) |
| **Solution** | `m = Σ(x−x̄)(y−ȳ) / Σ(x−x̄)²`, `b = ȳ − m·x̄` | Normal equation `β = (XᵀX)⁻¹Xᵀy` |
| **Interpretation** | Effect of one variable | Effect of each variable, others held constant |
| **Visualization** | Easy 2-D scatter + line | 3-D for 2 features, hard beyond that |
| **Risk of overfitting** | Very low | Higher when many features are used |
| **Multicollinearity issue** | ❌ Not applicable | ✅ Must be checked |
| **Best metric to compare** | R² | **Adjusted R²** |
| **Typical use** | Quick baseline, one clear driver | Real-world problems with several drivers |

### 🧪 Results of this notebook
⚠️ The two models use **different datasets**, so the scores show how each fit behaves here and should not be used to say that one method is "better" than the other.

In [ ]:
summary = pd.DataFrame({
    "Simple LR (placement data)": metrics_simple,
    "Multiple LR (synthetic data)": metrics_multi,
}).T
summary.insert(0, "Features", [1, 2])
summary.round(4)

## 📝 7. Notes on code improvements

While organising the notebook, a few things in the original code were corrected. 💡

| # | Original | Problem | Fix |
|---|---|---|---|
| 1 | `x_train.mean()` and `y_train.mean()` inside the `for` loop | The same averages were recalculated on every iteration (slow and cluttered) | Computed once before the loop |
| 2 | `fit` kept `x_train[i]` as 1-element arrays, so `m` and `b` were arrays like `[0.558]` | Results had an extra dimension and printed as arrays | `ravel()` → `m` and `b` are plain numbers |
| 3 | `x` and `y` were created twice (DataFrame, then `.values`) with two separate splits | Duplicate code, and the DataFrame version raised the *"X does not have valid feature names"* warning | One NumPy split used by both models |
| 4 | `r2adj = 1-((1-r2)*(40-1)/(40-1-1))` | `n=40` and `k=1` were hard-coded | Calculated from the data in `evaluate()` |
| 5 | Metrics typed in five separate cells | Repeated code | One reusable `evaluate()` function |
| 6 | Multiple regression: `y = lr.predict(x_test)` | **Overwrote the target variable `y`** | Predictions saved in `y_pred_multi` |
| 7 | Multiple regression: metrics were imported but never used | No evaluation of the model | Added `evaluate()` with `n_features=2` |
| 8 | `make_regression(...)` without a seed | Different data on every run | `random_state=42` |
| 9 | Placement data had to be present at `/content/` | Notebook crashed without the file | Falls back to synthetic data with a clear warning |

➕ **Added for learning:** true-vs-learned coefficient table, 3-D fitted plane and an actual-vs-predicted plot.

---
## 🏁 8. Summary

🔑 **Key takeaways**

1. 📈 **Linear regression** predicts a continuous target as a weighted sum of the inputs and finds the weights by **minimizing the sum of squared errors** (OLS).
2. 🎓 **Simple** linear regression has **one** feature and fits a **line**: `m = Σ(x−x̄)(y−ȳ)/Σ(x−x̄)²`, `b = ȳ − m·x̄`.
3. 🏠 **Multiple** linear regression has **two or more** features and fits a **plane / hyperplane**; each coefficient is the effect of that feature with the others held constant.
4. 🧑‍💻 Our from-scratch `Mylr` gives **exactly the same** slope and intercept as scikit-learn's `LinearRegression`, because both solve the same closed-form problem.
5. 📏 Evaluate on the **test set** with MAE, MSE, RMSE, R² and **adjusted R²** (use adjusted R² when comparing models with a different number of features).
6. ⚠️ Always check the assumptions: linearity, independent errors, constant variance, roughly normal residuals, and no multicollinearity.

🧭 **Quick guide — which one should I use?**

| Situation | Choice |
|---|---|
| One clear predictor, linear pattern | 🎓 Simple Linear Regression |
| Several factors influence the target | 🏠 Multiple Linear Regression |
| Relationship is curved | 🌀 Polynomial regression / non-linear models |
| Many correlated features or risk of overfitting | 🛡️ Ridge / Lasso regularization |

🚀 **What to explore next:**
- Implement **multiple** linear regression from scratch (Normal Equation) or train it with **Gradient Descent** — Batch, Stochastic and Mini-Batch (see the previous notebook)
- Polynomial regression, Ridge, Lasso and Elastic Net
- Feature scaling and checking multicollinearity with VIF

---
*Happy learning! 🎉*